In [ ]:
from pathlib import Path
import json

import numpy as np
import skimage
import sklearn
from skimage.feature import multiscale_basic_features
from skimage.io import imread
from skimage.segmentation import relabel_sequential
from sklearn.ensemble import RandomForestClassifier
from scipy.ndimage import gaussian_gradient_magnitude
from skl2onnx import to_onnx

from snap_to_edge import snap_labels_to_edge

In [ ]:
in_path = '/Users/david/Desktop/mito_segmentation_playground/20250207_yCJ009/'

image_subdirectory = 'tif'
mask_subdirectory = 'labels_sparse'

model_name = 'rf_yeast_mito_matrix_001'

do_snap_to_edge = True
snap_to_edge_radius = 1
snap_to_edge_ggm_sigma = 1.0

# kwargs to be passed to multiscale_basic_features
feature_fun_kwargs = {}

# how many pixels per class and image to sample
max_pixels_per_class = 100_000

# whether to sample pixels with replacement (potentially oversampling rare classes)
# if True, we will use max_pixels_per_class for each class
sample_with_replacement = True

# whether to relabel mask to have sequential 1, 2, ... labels
relabel_mask = True

# set to True if 0 means unannotated instead of bg
# classifier will be trained to predict idx - 1 (e.g. if 1 was bg & 2 cell, classifer will be trained with 0,1)
zero_is_unlabelled = True

In [ ]:
image_path = Path(in_path) / image_subdirectory
mask_path = Path(in_path) / mask_subdirectory

# get image-mask pairs
image_files = sorted(image_path.glob('*.tif'))
mask_files = sorted(mask_path.glob('*.tif'))

# show for verification
list(zip(image_files, mask_files))

In [ ]:
train_x = []
train_y = []

for idx in range(len(image_files)):

    # load image and mask
    img = imread(image_files[idx]).astype(float)
    mask = imread(mask_files[idx]).astype(int)
    
    # relabel if necessary
    if relabel_mask:
        mask, _, _ = relabel_sequential(mask)

    # refine masks with snap-to-edge
    if do_snap_to_edge:
        mask_ref = snap_labels_to_edge(mask, gaussian_gradient_magnitude(img, snap_to_edge_ggm_sigma), radius_morphology=snap_to_edge_radius)
    else:
        mask_ref = mask

    # calculate multiscale features (similar to ilastik, etc.)
    features = multiscale_basic_features(img, **feature_fun_kwargs)

    # flatten mask and features
    features_flat = features.reshape((-1, features.shape[-1]))
    mask_flat = mask_ref.ravel()

    for label in np.unique(mask_flat):

        # skip unlabelled 0 class
        if zero_is_unlabelled and label == 0:
            continue

        # pixels of class
        selection = np.flatnonzero(mask_flat == label)

        # sampled selection of those pixels
        if sample_with_replacement:
            selection_to_keep = np.random.choice(selection, max_pixels_per_class, replace=True)
        else:
            selection_to_keep = np.random.choice(selection, min(max_pixels_per_class, len(selection)), replace=False)

        # add to training x, y
        train_x.append(features_flat[selection_to_keep])
        # NOTE: when we ignore unlabelled 0, we subtract 1 from target
        # so the final prediction will be 0, 1, ... if valid input classes were 1, 2, ...
        train_y.append(mask_flat[selection_to_keep] - (1 if zero_is_unlabelled else 0))

# concat into one dataset
train_x = np.concatenate(train_x)
train_y = np.concatenate(train_y)

# fit RF
model = RandomForestClassifier(n_jobs=-1)
model.fit(train_x, train_y)

In [ ]:
# files to save
model_filename = model_name + '_model.onnx'
info_filename = model_name + '_info.json'

# info about model and feature extraction
info_dict = {
    'feature_fun_kwargs': feature_fun_kwargs,
    'model_file': model_filename,
    'sklearn_version': sklearn.__version__,
    'skimage_version': skimage.__version__
}

# save info to JSON
with open(Path(in_path) / info_filename, 'w') as f:
    json.dump(info_dict, f)

# save model to ONNX
onx = to_onnx(model, train_x[:1])
with open(Path(in_path) / model_filename, "wb") as f:
    f.write(onx.SerializeToString())

### Test snap-to-edge

Here, we load a single image + mask and show snap-to-edge results

In [ ]:
import napari

idx = 0

img = imread(image_files[idx]).astype(float)
mask = imread(mask_files[idx]).astype(int)
if relabel_mask:
    mask, _, _ = relabel_sequential(mask)

# refine mask with edge snap
mask_ref = snap_labels_to_edge(mask, gaussian_gradient_magnitude(img, snap_to_edge_ggm_sigma), radius_morphology=snap_to_edge_radius)
# relabel to start with higher indices (for visualization)
mask_ref, _, _ = relabel_sequential(mask_ref, np.max(mask) + 1)

if napari.current_viewer() is not None:
    napari.current_viewer().close()

viewer = napari.view_image(img)
viewer.add_labels(mask)
viewer.add_labels(mask_ref)

### Test segmentation

In [ ]:
idx = 1

img = imread(image_files[idx]).astype(np.float32)
mask = imread(mask_files[idx]).astype(int)
if relabel_mask:
    mask, _, _ = relabel_sequential(mask)

mask_ref = snap_labels_to_edge(mask, gaussian_gradient_magnitude(img, snap_to_edge_ggm_sigma), radius_morphology=snap_to_edge_radius)

features = multiscale_basic_features(img, **feature_fun_kwargs)
features_flat = features.reshape((-1, features.shape[-1]))


In [ ]:
# predict with sklearn model in memory
mask_pred = model.predict(features_flat).reshape(img.shape)

**Alternative:** Predict with saved ONNX model

In [ ]:
import onnxruntime as rt

sess = rt.InferenceSession(Path(in_path) / model_filename, providers=["CPUExecutionProvider"])
input_name = sess.get_inputs()[0].name
label_name = sess.get_outputs()[0].name
pred_onx = sess.run([label_name], {input_name: features_flat.astype(np.float32)})[0]

mask_pred = pred_onx.reshape(img.shape)

In [ ]:
import napari

# relabel to get different colors than GT mask in visualization
mask_pred, _, _ = relabel_sequential(mask_pred, np.max(mask) + 1)

refine_predicted = True
if refine_predicted:
    mask_pred = snap_labels_to_edge(mask_pred, gaussian_gradient_magnitude(img, snap_to_edge_ggm_sigma), radius_morphology=snap_to_edge_radius)

if napari.current_viewer() is not None:
    napari.current_viewer().close()

viewer = napari.view_image(img)
viewer.add_labels(mask_ref)
viewer.add_labels(mask_pred)